# Data Description
- Author: RUEI-HSIANG, CHANG（B12303030；AI 協助）
- Date first created: 20261019
- Background: 依使用者確認範圍，整理四類登記者蒐集缺漏與新官方人物頁線索；不產生研究結果。
- Unit of observation: master=一筆登記；gaps=候選人×欄位；profiles=候選人×內政部名錄命中；summary=公職×欄位×狀態。
- Sample period: 2026登記資料，既有2026-10-17/18蒐集與2026-10-19新存取。
- Known issues: 保留全部登記者而非最終資格樣本；背景、平台與觀察窗口未完整。跨表同縣市同姓名僅線索；學歷原文未獨立驗證。未找到不表示不存在，缺值不補零。資格原始標記保留但本次審定正文重新核對仍待完成。
- Definition for each variable:
  - 主表 candidate_key/姓名/公職/縣市/選區為原名冊欄位；party_registration_raw保留原推薦政黨；registration_source/file/page為登記來源。qualification_inherited保留上游標記，qualification_audit為本次複核狀態；sample_inclusion仍待組內決定；directory_lead_count為同縣市同姓名名錄命中數；*_status對應欄位清單狀態；updated_date為盤點日期。
  - 欄位清單 field為盤點項目；status區分未查、既有待複核、來源線索與未取得；existing_value為既有值或原文段落數，不作自動驗證；evidence為可取得之來源連結；note說明限制；next_source為下次查找方向。
  - 來源表 directory_key/profile_url為內政部名錄鍵與網址；match_status固定待身分核對；education_raw/experience_raw為官方頁標籤原文；http_status/page_contains_name/source_file/sha256/capture_error/captured_date為原始存取證據。
  - 摘要 count為該公職、欄位、狀態筆數；不是缺失率的跨欄位加總。
- Source data: 見處理Notebook固定 INPUTS（6份既有CSV及新capture_manifest.json），逐檔hash鎖定。
- Processing code: `data/code/prepare_candidate_inventory_20261019.ipynb`
- Role: collection-inventory（不作分析、train/test）
- Verification status: clean kernel執行；schema、唯一鍵、四類筆數、來源hash、CSV重讀逐值相等；不代表資料完整或身分驗證完成。
- Data path / SHA-256 (master): `data/processed/candidate_master_inventory_20261019.csv` / `1c5b8e716033949518a71c8972c21b3671de9487e25563d7c37f23b6ceb03474`；1583列。
- Data path / SHA-256 (gaps): `data/processed/candidate_field_inventory_20261019.csv` / `89bf3771afe5d0faf2c2e8353ab208a8459dc8b17e336c3da1f4b251473b3f7b`；31660列。
- Data path / SHA-256 (profiles): `data/processed/candidate_profile_leads_20261019.csv` / `52b723bdeea132df0f5ef9f9ac97594993784ba84edb796c82f0959d63236a3f`；692列。
- Data path / SHA-256 (summary): `data/processed/candidate_inventory_summary_20261019.csv` / `78dfdb83eac1197585deb6bc82e0e987c7b45fdbfe8c0b7f626ab92809ba242f`；108列。
- Downstream analysis input: 尚無；這些檔案作為蒐集工作單。

版本修訂：主表8筆政黨名稱移除來源PDF排版換行，一筆登記者占一個實體文字行；原始推薦政黨用字保留，上游CSV／PDF不變。舊主表hash 440336dcfee44dafcfb926dbdb21950687f5faca9efde430c6fff4645cc23502 已由本次版本取代。


In [1]:
from pathlib import Path
import hashlib,json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()


## Change for Your Own Data
依已保存檔案的固定hash及schema讀取，全欄字串，空白保留。

In [2]:
SPECS={'master': {'path': 'data/processed/candidate_master_inventory_20261019.csv', 'rows': 1583, 'columns': ['candidate_key', 'candidate_name', 'office_type', 'county_city', 'electoral_district', 'party_registration_raw', 'registration_source', 'registration_source_file', 'registration_source_page', 'qualification_inherited', 'qualification_audit', 'sample_inclusion', 'directory_lead_count', 'updated_date', 'sex_status', 'birth_date_status', 'education_status', 'incumbency_status', 'first_time_this_office_status', 'first_time_any_office_status', 'profile_status', 'website_status', 'Facebook_status', 'Instagram_status', 'Threads_status', 'posts_Facebook_status', 'posts_Instagram_status', 'posts_Threads_status', 'followers_status', 'policy_text_status', 'visual_status', 'political_family_status', 'administrative_experience_status', 'qualification_status'], 'sha256': '1c5b8e716033949518a71c8972c21b3671de9487e25563d7c37f23b6ceb03474', 'checks': 'schema, exact read-back, declared keys; not analysis-ready'}, 'gaps': {'path': 'data/processed/candidate_field_inventory_20261019.csv', 'rows': 31660, 'columns': ['candidate_key', 'candidate_name', 'office_type', 'county_city', 'field', 'status', 'existing_value', 'evidence', 'note', 'next_source', 'updated_date'], 'sha256': '89bf3771afe5d0faf2c2e8353ab208a8459dc8b17e336c3da1f4b251473b3f7b', 'checks': 'schema, exact read-back, declared keys; not analysis-ready'}, 'profiles': {'path': 'data/processed/candidate_profile_leads_20261019.csv', 'rows': 692, 'columns': ['candidate_key', 'candidate_name', 'county_city', 'directory_key', 'profile_url', 'match_status', 'education_raw', 'experience_raw', 'http_status', 'page_contains_name', 'source_file', 'sha256', 'capture_error', 'captured_date'], 'sha256': '52b723bdeea132df0f5ef9f9ac97594993784ba84edb796c82f0959d63236a3f', 'checks': 'schema, exact read-back, declared keys; not analysis-ready'}, 'summary': {'path': 'data/processed/candidate_inventory_summary_20261019.csv', 'rows': 108, 'columns': ['office_type', 'field', 'status', 'count'], 'sha256': '78dfdb83eac1197585deb6bc82e0e987c7b45fdbfe8c0b7f626ab92809ba242f', 'checks': 'schema, exact read-back, declared keys; not analysis-ready'}}
data={}
for key,s in SPECS.items():
 p=ROOT/s['path'];assert sha(p)==s['sha256']
 data[key]=pd.read_csv(p,dtype=str,keep_default_na=False)
 assert list(data[key].columns)==s['columns'] and len(data[key])==s['rows']


## Summary

In [3]:
display(data['master'].groupby('office_type').size().rename('登記筆數').to_frame())
print('來源線索',len(data['profiles']),'學歷原文非空',sum(data['profiles'].education_raw!=''))

,登記筆數
office_type,
直轄市議員,610
直轄市長,23
縣（市）議員,892
縣（市）長,58


來源線索 692 學歷原文非空 615


### Sample

In [4]:
for key,df in data.items():
 print(key);display(df.head(10))

master


,candidate_key,candidate_name,office_type,county_city,electoral_district,party_registration_raw,registration_source,registration_source_file,registration_source_page,qualification_inherited,...,Threads_status,posts_Facebook_status,posts_Instagram_status,posts_Threads_status,followers_status,policy_text_status,visual_status,political_family_status,administrative_experience_status,qualification_status
0,2026-C1-001,郭璽,直轄市長,臺北市,,台灣麻將最大黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
1,2026-C1-002,沈伯洋,直轄市長,臺北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
2,2026-C1-003,蕭文乾,直轄市長,臺北市,,臺灣SoR無法黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
3,2026-C1-004,唐新民,直轄市長,臺北市,,三勢團結促進聯盟,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
4,2026-C1-005,蔣萬安,直轄市長,臺北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
5,2026-C1-006,林志成,直轄市長,臺北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
6,2026-C1-007,蘇巧慧,直轄市長,新北市,,民主進步黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,部分原文已保存，尚未政策編碼,未查找,未查找,未查找,官方內文待重新核對
7,2026-C1-008,李四川,直轄市長,新北市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
8,2026-C1-009,蘇輝湟,直轄市長,新北市,,無,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對
9,2026-C1-010,張善政,直轄市長,桃園市,,中國國民黨,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...,data/raw/2026_metropolitan_registration.pdf,1,approved_2026-10-16,...,尚未驗證，不代表無帳號,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,未取得，不能視為零,待補查原文,未查找,未查找,未查找,官方內文待重新核對


gaps


,candidate_key,candidate_name,office_type,county_city,field,status,existing_value,evidence,note,next_source,updated_date
0,2026-C1-001,郭璽,直轄市長,臺北市,sex,已有資料，待來源複核,男,https://db.cec.gov.tw/,dob_corroborated_by_media,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
1,2026-C1-001,郭璽,直轄市長,臺北市,birth_date,已有資料，待來源複核,1958-08-31,https://db.cec.gov.tw/,CEC_name_and_birth_date_corroborated_by_CNA,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
2,2026-C1-001,郭璽,直轄市長,臺北市,education,已有資料，待來源複核,大學,,歷史公報或媒體補充，非2026學歷認證,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
3,2026-C1-001,郭璽,直轄市長,臺北市,incumbency,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
4,2026-C1-001,郭璽,直轄市長,臺北市,first_time_this_office,既有搜尋後仍缺，待補查,,,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
5,2026-C1-001,郭璽,直轄市長,臺北市,first_time_any_office,已有資料，待來源複核,0,https://db.cec.gov.tw/,0=有過參選證據；空白不表示首次。此處沿用既有表。,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
6,2026-C1-001,郭璽,直轄市長,臺北市,profile,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
7,2026-C1-001,郭璽,直轄市長,臺北市,website,既有搜尋後仍缺，待補查,,,,議會／政府人物頁、歷屆選舉公報；身分與時間核對,2026-10-19
8,2026-C1-001,郭璽,直轄市長,臺北市,Facebook,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19
9,2026-C1-001,郭璽,直轄市長,臺北市,Instagram,尚未驗證，不代表無帳號,,,,官方人物頁／競選網站交叉連結,2026-10-19


profiles


,candidate_key,candidate_name,county_city,directory_key,profile_url,match_status,education_raw,experience_raw,http_status,page_contains_name,source_file,sha256,capture_error,captured_date
0,2026|直轄市議員|臺北市第1選舉區|鍾佩玲,鍾佩玲,臺北市,ER11112AA00008,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,國立政治大學經營管理碩士\r\n世新大學口語傳播學系,壹電視新聞主播\r\n三立新聞主播\r\n臺灣新動力智庫教育文化組召集人\r\n臺灣綠色文化...,200,True,data/raw/inventory_20261019/ER11112AA00008.html,b5a97f9297fcde97cf6d9a597c599a393f3e26e0258833...,,2026-10-19
1,2026|直轄市議員|臺北市第1選舉區|陳賢蔚,陳賢蔚,臺北市,ER11112AA00002,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,國立臺灣海洋大學工學學士\r\n臺北市立復興高級中學,國立臺灣大學政治學系碩士生\r\n亞太自由婦女協會理事\r\n亞洲政經與和平交流協會理事\r...,200,True,data/raw/inventory_20261019/ER11112AA00002.html,2383c1bf70f9b4369613a1501bb0b0dc47c28e740b4c24...,,2026-10-19
2,2026|直轄市議員|臺北市第1選舉區|林世宗,林世宗,臺北市,ER11112AA00005,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,,,200,True,data/raw/inventory_20261019/ER11112AA00005.html,a3decfd80729d3888b6a975d64f49f34d811858e67261d...,,2026-10-19
3,2026|直轄市議員|臺北市第1選舉區|黃瀞瑩,黃瀞瑩,臺北市,ER11112AA00010,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,倫敦大學金匠學院政治傳播碩士\r\n天主教輔仁大學新聞傳播學系學士\r\n臺北市立明倫高中\...,臺北市政府副發言人\r\n黃珊珊副市長辦公室秘書\r\n臺北市政府媒體事務組\r\nETto...,200,True,data/raw/inventory_20261019/ER11112AA00010.html,d147843a665e68d235d023a26e391078411e0d9f35ade1...,,2026-10-19
4,2026|直轄市議員|臺北市第1選舉區|陳慈慧,陳慈慧,臺北市,ER11112AA00007,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,開南大學應用日語學系\r\n泰北高中\r\n士林區福安國中\r\n士林區社子國小,陽明山國際青商會副會長\r\n台灣棒球聯盟理事長,200,True,data/raw/inventory_20261019/ER11112AA00007.html,8aa78ea7e6bbee38032effb5cbef0dea9270d0de187167...,,2026-10-19
5,2026|直轄市議員|臺北市第1選舉區|陳重文,陳重文,臺北市,ER11112AA00006,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,國立體育大學體研所博士\r\n國立體育大學體育推廣學系碩士,臺北市立大學競技研究所、陸上系技擊系助理教授\r\n臺北市健美健身運動協會 理事長\r\n臺...,200,True,data/raw/inventory_20261019/ER11112AA00006.html,7fc144128866cdd806d73e258906710d5449c18bafd0da...,,2026-10-19
6,2026|直轄市議員|臺北市第1選舉區|林杏兒,林杏兒,臺北市,ER11112AA00003,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,銘傳大學公共事務學系碩士\r\n福安國中\r\n社子國小,中國國民黨副發言人\r\n台北市七星獅子會會長\r\n北投民防大隊副大隊長,200,True,data/raw/inventory_20261019/ER11112AA00003.html,b5f2f5a6ce7608aeb76a61cafb66ccf84ba8181f3ce248...,,2026-10-19
7,2026|直轄市議員|臺北市第1選舉區|張斯綱,張斯綱,臺北市,ER11112AA00011,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,國立政治大學東亞所碩士\r\n輔仁大學法律學系財經組,中國國民黨發言人 \r\n中國國民黨文傳會副主委\r\n盛弘醫藥股份有限公司副總經理兼發言人...,200,True,data/raw/inventory_20261019/ER11112AA00011.html,05716312c123fbdefc95420bd38b230ea4d4630257712f...,,2026-10-19
8,2026|直轄市議員|臺北市第1選舉區|汪志冰,汪志冰,臺北市,ER11112AA00004,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,美國東密西根大學文學碩士\r\n東吳大學德文系\r\n文藻外語學院德文科,中華民國全國總工會秘書長\r\n經濟部航太工業推動小組經理\r\n經濟部投資業務處專員\r\...,200,True,data/raw/inventory_20261019/ER11112AA00004.html,f094e3526171433c062cef97a6399d4fed4298906d7fa5...,,2026-10-19
9,2026|直轄市議員|臺北市第1選舉區|林延鳳,林延鳳,臺北市,ER11112AA00001,https://www.moi.gov.tw/LocalOfficial_Content.a...,county_name_match_only_pending_identity,臺灣大學生物產業傳播暨發展學系碩士,八大電視台行銷公關\r\n民進黨婦女部專員\r\n台北市友善空間協會秘書長,200,True,data/raw/inventory_20261019/ER11112AA00001.html,b6ba3b23d4e62f36ece056b3d663ef257e6b62d4031728...,,2026-10-19


summary


,office_type,field,status,count
0,直轄市議員,Facebook,未查找,610
1,直轄市議員,Instagram,未查找,610
2,直轄市議員,Threads,未查找,610
3,直轄市議員,administrative_experience,未查找,610
4,直轄市議員,birth_date,未查找,610
5,直轄市議員,education,找到名錄線索，待跨表身分核對,308
6,直轄市議員,education,未查找,302
7,直轄市議員,first_time_any_office,未查找,610
8,直轄市議員,first_time_this_office,未查找,610
9,直轄市議員,followers,未查找,610


### Summary Stats

In [5]:
display(pd.crosstab(data['gaps'].field,data['gaps'].status))
display(data['profiles'].groupby(['http_status','page_contains_name']).size().rename('N').to_frame())
for key,df in data.items():
 print(key,'空白欄位數');display((df=='').sum().to_frame('N'))

status,官方內文待重新核對,官方連結已保存，帳號內容待核對,尚未驗證，不代表無帳號,已有資料，待來源複核,待補查原文,找到名錄線索，待跨表身分核對,既有搜尋後仍缺，待補查,未取得，不能視為零,未查找,部分原文已保存，尚未政策編碼
field,,,,,,,,,,
Facebook,0,8,73,0,0,0,0,0,1502,0
Instagram,0,7,74,0,0,0,0,0,1502,0
Threads,0,2,79,0,0,0,0,0,1502,0
administrative_experience,0,0,0,0,0,0,0,0,1583,0
birth_date,0,0,0,55,0,0,26,0,1502,0
education,0,0,0,79,0,692,2,0,810,0
first_time_any_office,0,0,0,56,0,0,25,0,1502,0
first_time_this_office,0,0,0,21,0,0,60,0,1502,0
followers,0,0,0,0,0,0,0,81,1502,0


,,N
http_status,page_contains_name,
200,True,692


master 空白欄位數


,N
candidate_key,0
candidate_name,0
office_type,0
county_city,0
electoral_district,81
party_registration_raw,0
registration_source,0
registration_source_file,0
registration_source_page,0
qualification_inherited,0


gaps 空白欄位數


,N
candidate_key,0
candidate_name,0
office_type,0
county_city,0
field,0
status,0
existing_value,29730
evidence,27766
note,27485
next_source,0


profiles 空白欄位數


,N
candidate_key,0
candidate_name,0
county_city,0
directory_key,0
profile_url,0
match_status,0
education_raw,77
experience_raw,89
http_status,0
page_contains_name,0


summary 空白欄位數


,N
office_type,0
field,0
status,0
count,0


### Assertions

In [6]:
assert data['master'].candidate_key.is_unique
assert len(data['master'])==1583
assert not data['gaps'].duplicated(['candidate_key','field']).any()
assert set(data['profiles'].candidate_key)<=set(data['master'].candidate_key)
for key,s in SPECS.items():assert sha(ROOT/s['path'])==s['sha256']
print('通過：固定版本、schema、重讀、鍵與關聯檢查；未認證完整背景與最終候選人樣本')
assert not data['master'].party_registration_raw.str.contains(r'[\r\n]',regex=True).any()
assert len((ROOT/SPECS['master']['path']).read_text(encoding='utf-8-sig').splitlines())==1584


通過：固定版本、schema、重讀、鍵與關聯檢查；未認證完整背景與最終候選人樣本
